# Retailrocket Funnel Analysis

How does shopping conversion change over time?

Deduplicate shopping events and use visitor-level and visitor-item metrics to study conversion and returning visitors.

In [ ]:
import pandas as pd

events_raw = pd.read_csv("data/events.csv.gz")
events_raw["time"] = pd.to_datetime(events_raw["timestamp"], unit="ms")

events_raw.head()

In [ ]:
print(events_raw.shape)
events_raw.describe(include="all")

In [ ]:
for col_name in ["visitorid", "itemid", "transactionid"]:
    vc = events_raw[col_name].value_counts()
    print(f"col: {col_name} unique = {len(vc)} top = {vc.index[0]} freq={vc.iloc[0]}")

In [ ]:
events_raw["visitorid_str"] = events_raw["visitorid"].astype("str")
events_raw.describe(include="all")

In [ ]:
events_raw.dtypes

In [ ]:
events_raw["transactionid"].astype("float64")

In [ ]:
events_raw.duplicated().sum()

In [ ]:
events = events_raw.drop_duplicates()

print(len(events_raw), len(events))

In [ ]:
n_transaction_rows_without_id = events[events["event"] == "transaction"]["transactionid"].isna().sum()

n_other_rows_with_id = events[events["event"] != "transaction"]["transactionid"].notna().sum()

In [ ]:
n_transaction_rows = len(events[events["event"] == "transaction"])
n_transactions = events[events["event"] == "transaction"]["transactionid"].nunique()

In [ ]:
per_transaction = events.groupby("transactionid").agg(
    rows=("visitorid", "size"),
    visitors=("visitorid", "nunique"),
    moments=("timestamp", "nunique"),
)

print(per_transaction["visitors"].max(), per_transaction["moments"].max())
per_transaction.sort_values("moments", ascending=False).head()

In [ ]:
events["is_view"] = events["event"]=="view"
events["is_cart"] = events["event"]=="addtocart"
events["is_purchase"] = events["event"]=="transatcion"


visitors = events.groupby("visitorid").agg(
    events=("event", "size"),
    views=("is_view", "sum"),
    carts=("is_cart", "sum"),
    purchases=("is_purchase", "sum"),
    items=("itemid", "nunique")
)
print(visitors.head(5))

In [ ]:
events.describe()

In [ ]:
monthly = events.groupby(events["time"].dt.month).agg(
    views=("is_view", "sum"),
    carts=("is_cart", "sum"),
    purchases=("is_purchase", "sum")
)

monthly["view_to_cart"] = monthly["carts"] / monthly["views"]
monthly["cart_to_purchase"] = monthly["purchases"] / monthly["carts"]
monthly["view_to_purchase"] = monthly["purchases"] / monthly["views"]

In [ ]:
events["first_seen"] = events.groupby("visitorid")["time"].transform("min")

events["days_since_first"] = (
    events["time"] - events["first_seen"]
).dt.days

events["is_returning"] = events["days_since_first"] >= 1

In [ ]:
events["view_time"] = events["time"].where(events["event"] == "view")
events["cart_time"] = events["time"].where(events["event"] == "addtocart")
events["purchase_time"] = events["time"].where(events["event"] == "transaction")

events[["visitorid", "itemid", "event", "view_time", "cart_time", "purchase_time"]].head()

In [ ]:
pairs = events.groupby(["visitorid", "itemid"]).agg(
    first_view=("view_time", "min"),
    first_cart=("cart_time", "min"),
    first_purchase=("purchase_time", "min")
)

In [ ]:
pairs["view_to_cart"] = pairs["first_cart"] - pairs["first_view"]
pairs["cart_to_purchase"] = pairs["first_purchase"] - pairs["first_cart"]
pairs["view_to_purchase"] = pairs["first_purchase"] - pairs["first_view"]

In [ ]:
ai_disclosure = "..."